# 1. Download and Process Sweep Data
This cell connects to the W&B project `PCS_ET_v22` and specifically targets the sweeps `04ed65bp` and `ritttar1`. 

For each run, it performs the following:
1.  **Scans the training history** to find the epoch where `accuracy_validation` was maximized (Best Model Selection).
2.  **Extracts the Test metrics** (`accuracy_test` and `c_accuracy_test`) corresponding to that specific epoch.
3.  **Saves the results** to `PCS_ET_v22_results.csv` for further analysis.

In [1]:
import wandb
import pandas as pd
import numpy as np

# 1. Setup API
api = wandb.Api()
entity = "luis-perdigao-instituto-superior-t-cnico"
project = "PCS_ET_v22"
target_sweeps = ["lkhfhl2w"] # UPDATED SWEEP ID

print(f"Connecting to {entity}/{project}...")

data_list = []

# 2. Iterate through the specific sweeps
for sweep_id in target_sweeps:
    print(f"Fetching runs from sweep: {sweep_id}...")
    try:
        sweep = api.sweep(f"{entity}/{project}/{sweep_id}")
        runs = sweep.runs
    except Exception as e:
        print(f"Error accessing sweep {sweep_id}: {e}")
        continue

    print(f"  Found {len(runs)} runs. Scanning history...")

    for i, run in enumerate(runs):
        # A. Get Config
        config = {k: v for k, v in run.config.items() if not k.startswith('_')}
        
        # B. Fetch History
        # We need validation accuracy to pick the best epoch, and then we save BOTH val and test scores from that epoch
        keys = ["accuracy_validation", "c_accuracy_validation", "accuracy_test", "c_accuracy_test"]
        history = pd.DataFrame(run.scan_history(keys=keys))
        
        # Defaults
        val_rank_acc = np.nan
        val_class_acc = np.nan
        test_rank_acc = np.nan
        test_class_acc = np.nan
        
        if not history.empty and "accuracy_validation" in history.columns:
            # 1. Find the index (epoch) where VALIDATION accuracy was max
            best_epoch_idx = history["accuracy_validation"].idxmax()
            
            # 2. Retrieve VALIDATION scores from that epoch
            val_rank_acc = history.loc[best_epoch_idx, "accuracy_validation"]
            if "c_accuracy_validation" in history.columns:
                val_class_acc = history.loc[best_epoch_idx, "c_accuracy_validation"]

            # 3. Retrieve TEST scores from that epoch
            if "accuracy_test" in history.columns:
                test_rank_acc = history.loc[best_epoch_idx, "accuracy_test"]
            if "c_accuracy_test" in history.columns:
                test_class_acc = history.loc[best_epoch_idx, "c_accuracy_test"]
        else:
            # Fallback
            test_rank_acc = run.summary.get("accuracy_test", np.nan)
            test_class_acc = run.summary.get("c_accuracy_test", np.nan)
            val_rank_acc = run.summary.get("accuracy_validation", np.nan)
            val_class_acc = run.summary.get("c_accuracy_validation", np.nan)

        # C. Build Row
        data_list.append({
            "run_name": run.name,
            "sweep_id": sweep_id,
            "backbone": config.get("backbone", "unknown"),
            "num_ft_blocks": config.get("num_ft_blocks", 0),
            "seed": config.get("seed", np.nan), # ADDED SEED EXTRACTION
            "val_rank_acc": val_rank_acc,
            "val_class_acc": val_class_acc,
            "test_rank_acc": test_rank_acc,
            "test_class_acc": test_class_acc
        })

# 3. Save to CSV
filename = "PCS_ET_v22_results.csv"
df = pd.DataFrame(data_list)
df.to_csv(filename, index=False)
print(f"Done! Saved {len(df)} runs to '{filename}'.")

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: luis-perdigao (luis-perdigao-instituto-superior-t-cnico) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Connecting to luis-perdigao-instituto-superior-t-cnico/PCS_ET_v22...
Fetching runs from sweep: lkhfhl2w...
  Found 150 runs. Scanning history...
Done! Saved 150 runs to 'PCS_ET_v22_results.csv'.


# 2. Generate LaTeX Table
This cell reads the processed `PCS_ET_v22_results.csv` file and formats it into a LaTeX table for the paper.

It performs the following operations:
1.  **Cleans backbone names** for better readability (e.g., changing `dinov3_vitb16` to `DINOv3`).
2.  **Pivots the data** to organize rows by **Backbone** and columns by **Unfreeze Depth** (0, 1, 4, etc.).
3.  **Formats values** as `Rank Acc / Class Acc` and highlights the best result in each row in **bold**.

In [5]:
import pandas as pd
import numpy as np

def generate_v22_table_patched(csv_path="PCS_ET_v22_results.csv", mode="test"):
    """
    Generates LaTeX table.
    mode: 'test' or 'val' (determines which columns to show)
    """
    # 1. Load Data
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        return "Error: CSV file not found. Run Cell 1 first."

    # Determine columns based on mode
    if mode == "val":
        col_rank = 'val_rank_acc'
        col_class = 'val_class_acc'
        caption_text = "Validation Accuracy"
    else:
        col_rank = 'test_rank_acc'
        col_class = 'test_class_acc'
        caption_text = "Test Accuracy"

    # 2. Clean Backbone Names
    # FIXED: Re-added vit_base_patch16_224 along with the others
    backbone_map = {
        'dinov3_vitb16': 'DINOv3',
        'deit3_base_patch16_224': 'DeiT III',
        'vit_base_patch16_224': 'ViT-Base', 
        'vit_base_patch16_clip_224': 'CLIP',
        'clip': 'CLIP',
        'vit_small': 'ViT-Small'
    }
    df['backbone'] = df['backbone'].map(backbone_map).fillna(df['backbone'])

    # 3. Filter for specific layers ONLY
    target_layers = [0, 1, 4, 8, 12]
    df = df[df['num_ft_blocks'].isin(target_layers)]

    # 4. Pivot Table
    pivot_rank = df.pivot_table(index='backbone', columns='num_ft_blocks', values=col_rank, aggfunc='mean') * 100
    pivot_class = df.pivot_table(index='backbone', columns='num_ft_blocks', values=col_class, aggfunc='mean') * 100

    # 5. Calculate Averages
    avg_rank = pivot_rank.mean(axis=0)
    avg_class = pivot_class.mean(axis=0)

    # 6. Generate LaTeX
    latex_str = []
    latex_str.append("\\begin{table*}[ht]")
    latex_str.append("\\centering")
    latex_str.append(f"\\caption{{{caption_text} (Rank / Class) for unfreeze depths 1, 4, 8, and 12. Results averaged over 10 seeds. Best performance per backbone is highlighted in bold.}}")
    latex_str.append(f"\\label{{tab:v22_results_{mode}}}")
    
    # Define columns
    cols = [c for c in target_layers if c in pivot_rank.columns]
    col_def = "l" + "c" * len(cols)
    
    latex_str.append(f"\\begin{{tabular}}{{{col_def}}}")
    latex_str.append("\\toprule")
    
    # Header
    header_vals = [str(c) if c != 0 else "Frozen" for c in cols]
    latex_str.append(f"Backbone & {' & '.join(header_vals)} \\\\")
    latex_str.append("\\midrule")
    
    # Body Rows
    for backbone in pivot_rank.index:
        row_values = []
        
        # Calculate max for THIS specific backbone (ROW) across all depths
        row_max_r = pivot_rank.loc[backbone].max()
        row_max_c = pivot_class.loc[backbone].max()
        
        for col in cols:
            val_r = pivot_rank.loc[backbone, col]
            val_c = pivot_class.loc[backbone, col]
            
            if np.isnan(val_r) or np.isnan(val_c):
                row_values.append("-")
            else:
                s_r = f"{val_r:.2f}"
                s_c = f"{val_c:.2f}"
                
                # Highlight Best in Backbone (ROW)
                if abs(val_r - row_max_r) < 1e-9: s_r = f"\\textbf{{{s_r}}}"
                if abs(val_c - row_max_c) < 1e-9: s_c = f"\\textbf{{{s_c}}}"
                
                row_values.append(f"{s_r} / {s_c}")
        
        latex_str.append(f"\\textbf{{{backbone}}} & {' & '.join(row_values)} \\\\")

    # Average Row
    latex_str.append("\\midrule")
    avg_values = []
    
    max_avg_r = avg_rank.max()
    max_avg_c = avg_class.max()

    for col in cols:
        if col in avg_rank:
            val_r = avg_rank[col]
            val_c = avg_class[col]
            
            s_r = f"{val_r:.2f}"
            s_c = f"{val_c:.2f}"
            
            # Highlight Best Average
            if abs(val_r - max_avg_r) < 1e-9: s_r = f"\\textbf{{{s_r}}}"
            if abs(val_c - max_avg_c) < 1e-9: s_c = f"\\textbf{{{s_c}}}"
            
            avg_values.append(f"{s_r} / {s_c}")
        else:
            avg_values.append("-")

    latex_str.append(f"\\textit{{Average}} & {' & '.join(avg_values)} \\\\")

    latex_str.append("\\bottomrule")
    latex_str.append("\\end{tabular}")
    latex_str.append("\\end{table*}")

    return "\n".join(latex_str)

# Run Examples
print("% --------- TEST TABLE ---------")
print(generate_v22_table_patched(mode="test"))
print("\n% --------- VALIDATION TABLE ---------")
print(generate_v22_table_patched(mode="val"))

% --------- TEST TABLE ---------
\begin{table*}[ht]
\centering
\caption{Test Accuracy (Rank / Class) for unfreeze depths 1, 4, 8, and 12. Results averaged over 10 seeds. Best performance per backbone is highlighted in bold.}
\label{tab:v22_results_test}
\begin{tabular}{lccccc}
\toprule
Backbone & Frozen & 1 & 4 & 8 & 12 \\
\midrule
\textbf{DINOv3} & 71.88 / 72.14 & 73.63 / 72.66 & 74.47 / 74.43 & 73.98 / 73.90 & \textbf{74.52} / \textbf{74.47} \\
\textbf{DeiT III} & 69.44 / 68.32 & 72.52 / 72.29 & \textbf{73.82} / \textbf{73.66} & 73.30 / 72.83 & 73.59 / 73.35 \\
\textbf{ViT-Base} & 70.33 / 70.47 & \textbf{73.53} / 73.01 & 73.17 / \textbf{73.04} & 72.84 / 72.65 & 72.65 / 72.67 \\
\midrule
\textit{Average} & 70.55 / 70.31 & 73.22 / 72.65 & \textbf{73.82} / \textbf{73.71} & 73.37 / 73.12 & 73.59 / 73.50 \\
\bottomrule
\end{tabular}
\end{table*}

% --------- VALIDATION TABLE ---------
\begin{table*}[ht]
\centering
\caption{Validation Accuracy (Rank / Class) for unfreeze depths 1, 4, 8, an